# CosyVoice2 Ultra Voice Generation Studio

Self-contained Colab launcher. Select **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. The notebook clones the project repository, checks out the required branch, installs dependencies, downloads official CosyVoice/CosyVoice2 assets if missing, uses an optional Google Drive cache, and launches Gradio automatically.

In [ ]:
# ============================================================
# 0. Configuration: no manual edits required
# ============================================================
from __future__ import annotations
from pathlib import Path
import os, sys, subprocess, shutil, textwrap, platform

REPO_URL = "https://github.com/zbock-earn/login.git"
BRANCH = "codex/build-production-quality-ai-text-to-speech-app"
WORKSPACE = Path("/content") if Path("/content").exists() else Path.cwd()
REPO_DIR = WORKSPACE / "login"
COSYVOICE_DIR = WORKSPACE / "CosyVoice"
DRIVE_CACHE_ROOT = Path("/content/drive/MyDrive/CosyVoice2_Ultra_Cache")
USE_GOOGLE_DRIVE_CACHE = True


def progress(message: str) -> None:
    """Print a clear Colab progress message."""
    print(f"✔ {message}", flush=True)


def run(command: list[str] | str, cwd: str | Path | None = None) -> None:
    """Run a command and stream progress-friendly output."""
    printable = " ".join(command) if isinstance(command, list) else command
    print(f"$ {printable}", flush=True)
    subprocess.check_call(command, cwd=str(cwd) if cwd else None, shell=isinstance(command, str))

progress(f"Python {platform.python_version()} on {platform.platform()}")
progress(f"Workspace: {WORKSPACE}")


## 1. Clone or update this repository

In [ ]:
# ============================================================
# 1. Clone repository and checkout required branch
# ============================================================
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    progress("Repository already exists; pulling latest changes...")
    run(["git", "fetch", "origin"], cwd=REPO_DIR)
    run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    progress("Cloning repository...")
    run(["git", "clone", REPO_URL, str(REPO_DIR)])
    run(["git", "checkout", BRANCH], cwd=REPO_DIR)

progress(f"Repository ready: {REPO_DIR}")


## 2. Detect project root automatically

In [ ]:
# ============================================================
# 2. Detect project root without assuming a fixed folder name
# ============================================================
def find_project_root(repo_dir: Path) -> Path:
    """Find the app root by searching for the CosyVoice2 Ultra app entrypoint."""
    candidates: list[Path] = []
    for path in repo_dir.rglob("app.py"):
        if any(part.startswith(".") for part in path.relative_to(repo_dir).parts):
            continue
        text = path.read_text(errors="ignore")[:2000]
        if "build_ui" in text or "CosyVoice2" in text:
            candidates.append(path.parent)
    if candidates:
        return sorted(candidates, key=lambda item: len(item.parts))[0]
    for path in repo_dir.rglob("pyproject.toml"):
        if any(part.startswith(".") for part in path.relative_to(repo_dir).parts):
            continue
        return path.parent
    raise RuntimeError("Unable to detect project root after cloning repository. No suitable app.py or pyproject.toml found.")

PROJECT_ROOT = find_project_root(REPO_DIR)
sys.path.insert(0, str(PROJECT_ROOT))
progress(f"Detected project root: {PROJECT_ROOT}")


## 3. Configure Google Drive model cache

In [ ]:
# ============================================================
# 3. Optional Google Drive cache for model weights and HF cache
# ============================================================
def configure_drive_cache() -> Path:
    """Mount Drive when available and return the model cache directory."""
    local_cache = PROJECT_ROOT / "models" / "CosyVoice2-0.5B"
    if not (Path("/content").exists() and USE_GOOGLE_DRIVE_CACHE):
        local_cache.mkdir(parents=True, exist_ok=True)
        progress(f"Using local model cache: {local_cache}")
        return local_cache
    try:
        from google.colab import drive  # type: ignore
        progress("Mounting Google Drive for persistent model cache...")
        drive.mount("/content/drive", force_remount=False)
        model_cache = DRIVE_CACHE_ROOT / "models" / "CosyVoice2-0.5B"
        hf_cache = DRIVE_CACHE_ROOT / "huggingface"
        model_cache.mkdir(parents=True, exist_ok=True)
        hf_cache.mkdir(parents=True, exist_ok=True)
        os.environ["HF_HOME"] = str(hf_cache)
        os.environ["HUGGINGFACE_HUB_CACHE"] = str(hf_cache / "hub")
        os.environ["COSYVOICE2_MODEL_DIR"] = str(model_cache)
        progress(f"Using Google Drive model cache: {model_cache}")
        return model_cache
    except Exception as exc:
        local_cache.mkdir(parents=True, exist_ok=True)
        os.environ["COSYVOICE2_MODEL_DIR"] = str(local_cache)
        progress(f"Google Drive cache unavailable ({exc}); using local cache: {local_cache}")
        return local_cache

MODEL_CACHE_DIR = configure_drive_cache()


## 4. Install system and project dependencies

In [ ]:
# ============================================================
# 4. Install dependencies automatically
# ============================================================
progress("Installing dependencies...")
if Path("/content").exists():
    run("apt-get update -y && apt-get install -y ffmpeg sox libsox-dev git git-lfs")
run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
requirements = PROJECT_ROOT / "requirements.txt"
if requirements.exists():
    run([sys.executable, "-m", "pip", "install", "-r", str(requirements)])
else:
    run([sys.executable, "-m", "pip", "install", "torch", "torchaudio", "gradio", "huggingface_hub", "modelscope", "librosa", "soundfile", "pydub", "psutil", "num2words", "numpy", "scipy", "rich"])
progress("Project dependencies installed.")


## 5. Clone official CosyVoice implementation if missing

In [ ]:
# ============================================================
# 5. Clone/install official CosyVoice implementation
# ============================================================
import importlib.util
if importlib.util.find_spec("cosyvoice") is None:
    if COSYVOICE_DIR.exists() and (COSYVOICE_DIR / ".git").exists():
        progress("Official CosyVoice repository already exists; pulling latest changes...")
        run(["git", "pull", "--ff-only"], cwd=COSYVOICE_DIR)
    else:
        if COSYVOICE_DIR.exists():
            shutil.rmtree(COSYVOICE_DIR)
        progress("Downloading official CosyVoice implementation...")
        run(["git", "clone", "--depth", "1", "https://github.com/FunAudioLLM/CosyVoice.git", str(COSYVOICE_DIR)])
    run([sys.executable, "-m", "pip", "install", "-e", str(COSYVOICE_DIR)])
else:
    progress("Official CosyVoice Python package already installed.")

sys.path.insert(0, str(COSYVOICE_DIR))
progress("CosyVoice implementation ready.")


## 6. Check GPU/RAM/VRAM and acceleration support

In [ ]:
# ============================================================
# 6. Hardware diagnostics for T4 / L4 / A100
# ============================================================
progress("Checking CUDA, GPU, RAM, and VRAM...")
import psutil, torch
from runtime_optimizer import configure_runtime
from utils import nvidia_smi
report = configure_runtime()
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("RAM GB:", round(psutil.virtual_memory().total / 1024**3, 2))
print("torch.compile supported:", report.torch_compile_supported)
print("Flash Attention available:", report.flash_attention_available)
print("Mixed precision:", report.mixed_precision)
print("\n".join(report.notes))
print(nvidia_smi())


## 7. Download CosyVoice2 model/tokenizer only if missing

In [ ]:
# ============================================================
# 7. Download/cache model weights and tokenizer
# ============================================================
progress("Downloading CosyVoice2 model/tokenizer if not already cached...")
# Re-import settings/cache after COSYVOICE2_MODEL_DIR is set.
import importlib
import settings, cache
importlib.reload(settings)
importlib.reload(cache)
model_path = cache.ensure_model()
progress(f"CosyVoice2 cache ready: {model_path}")
print("Cached file count:", len(list(Path(model_path).rglob("*"))))


## 8. Launch the Gradio Voice Generation Studio

In [ ]:
# ============================================================
# 8. Launch application automatically
# ============================================================
progress("Loading model and launching Studio...")
os.chdir(PROJECT_ROOT)
from app import main
main()
